# Step 6: Infer gender of the recovered forenames (Wiki-Gendersort)

Wiki-Gendersort is applied to the forenames recovered in step 5. Each distinct forename is classified only once
(identical result to one call per row).

**Input:** `data/derived/nan_ini_medline_names.csv` (output of `step05_recover_forenames.ipynb`)
**Output:** `data/derived/nan_ini_medline_names_and_gender.csv`

Requires Wiki-Gendersort: `git clone https://github.com/nicolasberube/Wiki-Gendersort.git` (a separate Python environment is recommended).

In [ ]:
import os
import pandas as pd

WG_DIR = 'path/to/Wiki-Gendersort'                # local clone of Wiki-Gendersort (adapt)
BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
DERIVED_DIR = f'{BASE_DIR}/data/derived'
IN_CSV = f'{DERIVED_DIR}/nan_ini_medline_names.csv'
OUT_CSV = f'{DERIVED_DIR}/nan_ini_medline_names_and_gender.csv'

os.chdir(WG_DIR)                      # Wiki-Gendersort reads NamesOut.txt from its own folder
from Wiki_Gendersort import wiki_gendersort
WG = wiki_gendersort()

In [ ]:
df = pd.read_csv(IN_CSV, dtype={'LastName': str, 'ForeName_pkg': str, 'gender': str, 'medline_name': str, 'forename': str})
print(f'{len(df):,} rows, {df.forename.notna().sum():,} with recovered forename')

In [ ]:
# classify every distinct forename once; rows without recovered forename keep gender = NaN
names = df['forename'].dropna().astype(str).unique()
lookup = {n: WG.assign(n) for n in names}
df['gender'] = df['forename'].map(lookup)
print(df['gender'].value_counts(dropna=False))

`NaN` = no gender could be recovered: the paper is not in the MEDLINE file, no author with that surname is on the paper, or MEDLINE's forename field is empty.

In [ ]:
print(df['gender'].value_counts(dropna=False, normalize=True))

Counts above are author × paper rows, not authors.

In [ ]:
df.to_csv(OUT_CSV, index=False)
print('written', OUT_CSV)